In [5]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

In [6]:
year = 2021

In [7]:
data_dir = Path("data")
INDIR = Path(f"../data/data_model/{year}")
OUTDIR_IMG = Path(f"../report/img/{year}")
OUTDIR_IMG.mkdir(parents=True, exist_ok=True)

In [8]:
state_file = INDIR / f"ENEM_SCORES_STATE_BRAZIL_CLUSTERS_{year}.csv"
df_state = pd.read_csv(state_file, sep=",")

state_clusters = {
    0: df_state[df_state['CLUSTER'] == 0].copy().reset_index(drop=True),
    1: df_state[df_state['CLUSTER'] == 1].copy().reset_index(drop=True),
    2: df_state[df_state['CLUSTER'] == 2].copy().reset_index(drop=True)
}

In [9]:
df_state.head()

,STATE,NUM_PARTICIPANTS,FAMILY_INCOME_SM_AVG,NATURAL_SCIENCES_SCORE_AVG,HUMANITIES_SCORE_AVG,LANGUAGES_SCORE_AVG,MATH_SCORE_AVG,ESSAY_SCORE_AVG,OVERALL_SCORE_AVG,CLUSTER
0,AC,12328.0,1.602943,470.478318,503.314406,486.900552,496.863546,606.685594,512.848483,0
1,AL,37261.0,1.579928,477.397362,505.964223,487.847304,517.739768,637.479402,525.285612,1
2,AM,41905.0,1.326470,462.690564,490.134645,475.850273,488.711454,576.426202,498.762628,0
3,AP,12464.0,1.662385,468.976845,501.532084,478.752399,492.902920,606.846919,509.802234,0
4,BA,169479.0,1.473967,480.302063,510.339846,492.215575,514.043935,629.168334,525.213951,1


In [10]:

url = "https://raw.githubusercontent.com/codeforamerica/click_that_hood/master/public/data/brazil-states.geojson"


In [11]:
score_cols = [c for c in df_state.columns if c.endswith("_MEDIA")]
cluster_colors = {0: "#ff0e0e", 1: "#1f77b4", 2: "#2ca02c"}

performance_labels = {0: "Low", 1: "Intermediate", 2: "High"}
performance_colors = {
    "Low": cluster_colors[0],
    "Intermediate": cluster_colors[1],
    "High": cluster_colors[2],
}

In [12]:
df_map = df_state.copy()
df_map["PERFORMANCE"] = df_map["CLUSTER"].map(performance_labels)

fig = px.choropleth(
    df_map,
    geojson=url,
    locations="STATE",
    featureidkey="properties.sigla",
    color="PERFORMANCE",
    color_discrete_map=performance_colors,
    category_orders={"PERFORMANCE": ["Low", "Intermediate", "High"]},
    title=f"Performance by State (ENEM {year})",
    labels={"PERFORMANCE": "Performance"},
    custom_data=[
        "STATE",
        "CLUSTER",
        "OVERALL_SCORE_AVG",
        "FAMILY_INCOME_SM_AVG",
        "NUM_PARTICIPANTS"
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "State: %{customdata[0]}<br>"
        "Cluster: %{customdata[1]}<br>"
        "Overall Average: %{customdata[2]:.2f}<br>"
        "Avg. Income: %{customdata[3]:.2f}<br>"
        "Participants: %{customdata[4]}<br>"
        "<extra></extra>"
    )
)

fig.show()

In [13]:
df_map = df_state.copy()

fig = px.choropleth(
    df_map,
    geojson=url,
    locations='STATE',
    featureidkey='properties.sigla',
    color='OVERALL_SCORE_AVG',
    color_continuous_scale='RdYlGn',
    title=f'Overall Average Score by State (ENEM {year})',
    labels={'OVERALL_SCORE_AVG': 'Overall Average Score'},
    custom_data=[
        'STATE',
        'OVERALL_SCORE_AVG',
        'FAMILY_INCOME_SM_AVG',
        'NUM_PARTICIPANTS'
    ]
)

fig.update_geos(fitbounds="locations", visible=False)

fig.update_traces(
    hovertemplate=(
        "State: %{customdata[0]}<br>"
        "Overall Average Score: %{customdata[1]:.2f}<br>"
        "Avg. Family Income: %{customdata[2]:.2f}<br>"
        "Participants: %{customdata[3]}<br>"
        "<extra></extra>"
    )
)
"""
path = OUTDIR_IMG / f"RENDIMENTO_GERAL_{year}.png"
fig.write_image(str(path), scale=2)
print(f"Image saved at: {path}")
"""
fig.show()